In [11]:
import time
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

tickers = pd.read_csv("../data/tickers.csv")
print("Companies:", len(tickers))
print(tickers["sector"].value_counts())

Companies: 56
sector
Tech              14
Financials        12
Consumer          10
Transport          9
Healthcare         8
Semiconductors     3
Name: count, dtype: int64


In [12]:
ENDPOINTS = [
    "income-statement",
    "balance-sheet-statement",
    "cash-flow-statement",
    "employee-count",
]

def fetch_and_cache(ticker, endpoint, limit=5):
    path = RAW_DIR / f"{ticker}_{endpoint}.json"

    if path.exists():
        return "cached"

    params = {"symbol": ticker, "apikey": API_KEY}
    if endpoint != "employee-count":
        params["limit"] = limit

    response = requests.get(f"{BASE}/{endpoint}", params=params)
    response.raise_for_status()
    data = response.json()

    if not data:
        return "empty"

    with open(path, "w") as f:
        json.dump(data, f)
    return "fetched"

In [13]:
results = []

for row in tickers.itertuples():
    for endpoint in ENDPOINTS:
        try:
            status = fetch_and_cache(row.ticker, endpoint)
        except Exception as e:
            status = f"failed: {type(e).__name__}"
        results.append({"ticker": row.ticker, "endpoint": endpoint, "status": status})
        if status == "fetched":
            time.sleep(0.3)

log = pd.DataFrame(results)
print(log["status"].value_counts())

status
fetched    224
Name: count, dtype: int64


In [14]:
problems = log[~log["status"].isin(["cached", "fetched"])]
print("Problem count:", len(problems))
problems

Problem count: 0


,ticker,endpoint,status


In [31]:
def combining_company_statement(statement_type):
    frames = []

    for row in tickers.itertuples():
        path = RAW_DIR / f"{row.ticker}_{statement_type}.json"

        with open(path, "r") as f:
            frames.append(pd.DataFrame(json.load(f)))

    return pd.concat(frames, ignore_index=True)

In [ ]:
income = combining_company_statement("income-statement")
balance = combining_company_statement("balance-sheet-statement")
cashflow = combining_company_statement("cash-flow-statement")

for name, df in [("income", income), ("balance", balance), ("cashflow", cashflow)]:
    print(name, df.shape)


income (280, 39)
balance (280, 61)
cashflow (280, 47)
